# 🏭 POC 4: Full Production Pipeline — End-to-End

**What you'll learn:**
- Complete CI/CD: Build → Test → Scan → Deploy Dev → Staging → Prod
- Pipeline Templates (reusable across teams)
- OPA Policy as Code (governance)
- Triggers (Git push, Cron, Webhook)
- Secrets management
- DORA metrics tracking

In [ ]:
# Step 1: Generate a Reusable CI Template
# Templates let ANY team reuse the same pipeline pattern
import yaml, os

ci_template = {
    "template": {
        "name": "Python ML CI Template",
        "identifier": "python_ml_ci_template",
        "versionLabel": "1.0",
        "type": "Stage",
        "spec": {
            "type": "CI",
            "spec": {
                "cloneCodebase": True,
                "execution": {
                    "steps": [
                        {"step": {"type": "Run", "name": "Install Deps",
                                  "spec": {"command": "pip install -r requirements.txt"}}},
                        {"step": {"type": "Run", "name": "Lint",
                                  "spec": {"command": "flake8 app/ --max-line-length=120"}}},
                        {"step": {"type": "Run", "name": "Test",
                                  "spec": {"command": "pytest tests/ -v --cov=app --cov-fail-under=70"}}},
                        {"step": {"type": "Run", "name": "Security Scan",
                                  "spec": {"command": "pip audit || true"}}},
                        {"step": {"type": "BuildAndPushDockerRegistry", "name": "Build & Push",
                                  "spec": {"connectorRef": "<+stage.variables.dockerConnector>",
                                           "repo": "<+stage.variables.dockerRepo>",
                                           "tags": ["<+pipeline.sequenceId>", "latest"]}}}
                    ]
                }
            },
            "variables": [
                {"name": "dockerConnector", "type": "String", "required": True},
                {"name": "dockerRepo", "type": "String", "required": True}
            ]
        }
    }
}

os.makedirs('.harness/templates', exist_ok=True)
with open('.harness/templates/ci-template.yaml', 'w') as f:
    yaml.dump(ci_template, f, default_flow_style=False, sort_keys=False)
print("✅ Created reusable CI template")
print("💡 Any team can now use: templateRef: python_ml_ci_template")

In [ ]:
# Step 2: OPA Policy as Code (Governance)
# These BLOCK pipelines that violate rules

no_friday_policy = '''# No deployments on Friday/weekends
package pipeline

deny[msg] {
    day := time.weekday(time.now_ns())
    day == "Friday"
    input.pipeline.stages[_].stage.spec.environment.environmentRef == "production"
    msg := "BLOCKED: No production deploys on Fridays!"
}
'''

require_approval = '''# Production must have approval
package pipeline

deny[msg] {
    stage := input.pipeline.stages[_].stage
    stage.spec.environment.environmentRef == "production"
    not has_approval
    msg := "BLOCKED: Production needs an Approval stage!"
}

has_approval {
    input.pipeline.stages[_].stage.type == "Approval"
}
'''

os.makedirs('.harness/policies', exist_ok=True)
for name, policy in [("no_friday_deploy", no_friday_policy), ("require_approval", require_approval)]:
    with open(f'.harness/policies/{name}.rego', 'w') as f:
        f.write(policy)
    print(f"✅ Created policy: {name}.rego")

print("\n💡 OPA policies are written in Rego language")
print("   They automatically BLOCK pipelines that violate rules")

In [ ]:
# Step 3: Full Production Pipeline
# CI Template → Dev → Staging → Approval → Canary Prod

def make_cd_stage(name, env, strategy="rolling"):
    steps = []
    rollback_type = "K8sRollingRollback"
    if strategy == "canary":
        steps = [
            {"step": {"type": "K8sCanaryDeploy", "name": "Canary 20%",
                      "spec": {"instanceSelection": {"type": "Percentage", "spec": {"percentage": 20}}}}},
            {"step": {"type": "ShellScript", "name": "Verify Canary",
                      "spec": {"shell": "Bash", "source": {"type": "Inline",
                               "spec": {"script": "sleep 30 && echo 'Canary healthy!'"}}}}},
            {"step": {"type": "K8sRollingDeploy", "name": "Full Rollout", "spec": {"skipDryRun": False}}},
            {"step": {"type": "K8sCanaryDelete", "name": "Cleanup", "spec": {}}}
        ]
        rollback_type = "K8sCanaryDelete"
    else:
        steps = [{"step": {"type": "K8sRollingDeploy", "name": "Deploy", "spec": {"skipDryRun": False}}}]
    
    return {"stage": {
        "name": name, "type": "Deployment",
        "spec": {
            "deploymentType": "Kubernetes",
            "service": {"serviceRef": "ml_sentiment_api"},
            "environment": {"environmentRef": env,
                            "infrastructureDefinitions": [{"identifier": f"{env}_k8s"}]},
            "execution": {
                "steps": steps,
                "rollbackSteps": [{"step": {"type": rollback_type, "name": "Rollback", "spec": {}}}]
            }
        }
    }}

full_pipeline = {
    "pipeline": {
        "name": "Full Production Pipeline",
        "identifier": "full_production_pipeline",
        "projectIdentifier": "ml_platform",
        "orgIdentifier": "default",
        "tags": {"team": "ml-platform", "tier": "production"},
        "stages": [
            # Stage 1: CI (uses template)
            {"stage": {
                "name": "CI Build & Test",
                "type": "CI",
                "template": {
                    "templateRef": "python_ml_ci_template",
                    "versionLabel": "1.0",
                    "templateInputs": {
                        "variables": [
                            {"name": "dockerConnector", "value": "dockerhub_connector"},
                            {"name": "dockerRepo", "value": "myuser/ml-sentiment-api"}
                        ]
                    }
                }
            }},
            # Stage 2: Deploy Dev (auto)
            make_cd_stage("Deploy Dev", "dev"),
            # Stage 3: Deploy Staging (auto)
            make_cd_stage("Deploy Staging", "staging"),
            # Stage 4: Approval
            {"stage": {
                "name": "Production Approval",
                "type": "Approval",
                "spec": {"execution": {"steps": [{"step": {
                    "type": "HarnessApproval", "name": "Approve Prod",
                    "spec": {"approvalMessage": "Dev ✅ Staging ✅ → Approve production?",
                             "approvers": {"userGroups": ["_project_all_users"], "minimumCount": 1}},
                    "timeout": "1d"
                }}]}}
            }},
            # Stage 5: Canary to Production
            make_cd_stage("Canary to Production", "production", "canary")
        ],
        "notificationRules": [{
            "name": "Slack Alerts",
            "pipelineEvents": [{"type": "PipelineFailed"}, {"type": "PipelineSuccess"}],
            "notificationMethod": {"type": "Slack", "spec": {"webhookUrl": "<+input>"}}
        }]
    }
}

os.makedirs('.harness/pipelines', exist_ok=True)
with open('.harness/pipelines/full-production.yaml', 'w') as f:
    yaml.dump(full_pipeline, f, default_flow_style=False, sort_keys=False)

print("✅ Created full production pipeline!")
print("\n📋 PIPELINE FLOW:")
print("   CI Template → Dev → Staging → Approval → Canary Prod")
print("   + OPA Policies + Slack Notifications + Auto Rollback")

In [ ]:
# Step 4: Pipeline Triggers

triggers = {
    "git_push": {
        "trigger": {
            "name": "On Push to Main",
            "identifier": "git_push_main",
            "enabled": True,
            "pipelineIdentifier": "full_production_pipeline",
            "source": {
                "type": "Webhook",
                "spec": {"type": "Github", "spec": {"type": "Push",
                         "spec": {"connectorRef": "github_connector",
                                  "payloadConditions": [{"key": "targetBranch", "operator": "Equals", "value": "main"}]}}}
            }
        }
    },
    "nightly_build": {
        "trigger": {
            "name": "Nightly Build",
            "identifier": "nightly_build",
            "enabled": True,
            "pipelineIdentifier": "full_production_pipeline",
            "source": {"type": "Scheduled", "spec": {"type": "Cron", "spec": {"expression": "0 2 * * *"}}}
        }
    }
}

os.makedirs('.harness/triggers', exist_ok=True)
for name, trigger in triggers.items():
    with open(f'.harness/triggers/{name}.yaml', 'w') as f:
        yaml.dump(trigger, f, default_flow_style=False, sort_keys=False)
    print(f"✅ Created trigger: {name}")

print("\n📋 TRIGGER TYPES:")
print("   1. Git Push  → Pipeline runs on every push to main")
print("   2. Cron      → Nightly build at 2 AM UTC")
print("   3. Webhook   → External system calls Harness API")
print("   4. Artifact  → New Docker image appears in registry")

In [ ]:
# Step 5: Smoke Test Script (used in pipeline)

smoke_test = '''#!/bin/bash
set -e
APP_URL="${1:-http://localhost:8000}"
echo "🧪 Smoke Tests: $APP_URL"

test_endpoint() {
    local url=$1 expected=$2 desc=$3
    status=$(curl -s -o /dev/null -w "%{http_code}" "$url" --max-time 10)
    if [ "$status" -eq "$expected" ]; then echo "  ✅ $desc"; else echo "  ❌ $desc (got $status)"; exit 1; fi
}

test_endpoint "$APP_URL/" 200 "Root"
test_endpoint "$APP_URL/health" 200 "Health"
test_endpoint "$APP_URL/health/ready" 200 "Readiness"
echo "✅ All smoke tests passed!"
'''

os.makedirs('scripts', exist_ok=True)
with open('scripts/smoke-test.sh', 'w') as f:
    f.write(smoke_test)
print("✅ Created scripts/smoke-test.sh")

In [ ]:
# Step 6: DORA Metrics — Measure Your Team's Performance

print("📊 DORA METRICS — What Top Teams Track")
print("=" * 60)

dora = [
    ("Deployment Frequency", "How often you deploy", "Multiple/day", "Once/month"),
    ("Lead Time for Changes", "Code commit → production", "< 1 hour", "> 1 month"),
    ("Change Failure Rate", "% of deploys causing failure", "< 5%", "> 45%"),
    ("Mean Time to Recover", "Time to fix production issue", "< 1 hour", "> 1 week"),
]

print(f"{'Metric':<25} {'What':<30} {'Elite':<15} {'Low'}")
print("-" * 80)
for m in dora:
    print(f"{m[0]:<25} {m[1]:<30} {m[2]:<15} {m[3]}")

print("\n💡 Harness tracks these automatically in Software Engineering Insights!")

## 🎯 POC 4 Summary — Full Production Pipeline

**Complete flow:**
```
Git Push → CI Template (lint+test+scan+build) → Dev → Staging → Approval → Canary Prod
         + OPA Policies (no Friday deploy, require approval)
         + Slack Notifications
         + Auto Rollback on failure
         + DORA Metrics tracking
```

**Files generated across all 4 POCs:**
| File | Purpose |
|------|---------|
| `app/main.py` | FastAPI ML API |
| `tests/test_app.py` | pytest tests |
| `Dockerfile` | Multi-stage container |
| `k8s/*.yaml` | Kubernetes manifests |
| `.harness/ci-pipeline.yaml` | CI pipeline |
| `.harness/cd-*.yaml` | CD pipelines (3 strategies) |
| `.harness/templates/` | Reusable templates |
| `.harness/policies/` | OPA governance |
| `.harness/triggers/` | Auto-triggers |
| `scripts/smoke-test.sh` | Post-deploy verification |

## 🎓 You are now a Harness Engineer! Next steps:
1. Sign up at https://app.harness.io
2. Install a Delegate in your K8s cluster
3. Copy these YAMLs into Harness Pipeline Studio
4. Get Harness Certified Developer certification!